# Leggere e scrivere CSV

Iniziamo con le funzionalità offerte dalla libreria standard.

In [1]:
import csv

## Leggere 

Iniziamo trascurando le intestazioni, lavorando per righe. L'idea è di aprire un *file* (che può anche essere una connessione di rete ottenuta con `urlopen`) e di "avvolgere" tale *file* con un **reader**.

In [2]:
with open('tesi.csv', encoding = 'utf-8') as infile:
  reader = csv.reader(infile)
  for n, row in enumerate(reader):
    if n == 5: break # leggiamo solo le prime 5 righe
    print(row)

['HANDLE', 'TITLE', 'AUTHOR', 'TYPE', 'YEAR', 'ABSTRACT', 'LANGUAGE', 'CITATION', 'FILE_NAME', 'FILE_FORMAT', 'FILE_VERSION', 'FILE_ACCESS', 'FILE_URL', 'LICENSE']
['https://hdl.handle.net/2434/956986', 'THE 2019 EP ELECTION CAMPAIGN AS SEEN FROM TWITTER', 'PAGANO, GIOVANNI', 'Tesi di dottorato', '2022', "The end of May 2019 saw European citizens go to the polls to choose a new European Parliament (EP). These elections marked a crucial moment for European democracy, as they took place in a context of significant challenges and upheavals, including the aftermath of the economic and financial crisis that had impacted the EU in the preceding decade, the refugee crisis that had tested the EU's capacity to manage migration and asylum, and the departure of the United Kingdom from the EU. These events had far-reaching implications for the political landscape of the EU, and the election campaign was shaped by the issues and concerns that they raised, as well as the parties and candidates who s

Invece di semplici *list* possiamo leggere dei *dict* in cui le chiavi siano le intestazioni di colonna. In questo caso usiamo un **DictReader**.

In [3]:
with open('tesi.csv', encoding = 'utf-8') as infile:
  reader = csv.DictReader(infile)
  for n, row in enumerate(reader):
    if n == 5: break # leggiamo solo le prime 5 righe
    print(row)

{'HANDLE': 'https://hdl.handle.net/2434/956986', 'TITLE': 'THE 2019 EP ELECTION CAMPAIGN AS SEEN FROM TWITTER', 'AUTHOR': 'PAGANO, GIOVANNI', 'TYPE': 'Tesi di dottorato', 'YEAR': '2022', 'ABSTRACT': "The end of May 2019 saw European citizens go to the polls to choose a new European Parliament (EP). These elections marked a crucial moment for European democracy, as they took place in a context of significant challenges and upheavals, including the aftermath of the economic and financial crisis that had impacted the EU in the preceding decade, the refugee crisis that had tested the EU's capacity to manage migration and asylum, and the departure of the United Kingdom from the EU. These events had far-reaching implications for the political landscape of the EU, and the election campaign was shaped by the issues and concerns that they raised, as well as the parties and candidates who sought to address them. Against this backdrop, the 2019 EP election campaign was marked by a high degree of 

Naturalmente è preferibile usare una espressione come `row['YEAR']` piuttosto che `row[4]` perché è più leggibile e più robusta rispetto a cambiamenti nell'ordine o numero di colonne. Ma è anche vero che i dizionari richiedono più memoria e sono probabilmente più lenti da costruire. 

Un modo di aggirare il problema è usare la prima riga per scoprire l'indice della colonna, e poi leggere per *list*.

In [4]:
with open('tesi.csv', encoding = 'utf-8') as infile:
  reader = csv.reader(infile)
  headers = next(reader)
  year_pos = headers.index('YEAR')
  for n, row in enumerate(reader):
    if n == 5: break # leggiamo solo le prime 5 righe
    year = row[year_pos]
    print(year) 

2022
2024
2020
2023
2020


Potremmo usare il precedente codice per contare il numero di record per anno. Usiamo un dizionario `anno2num` per memorizzare il conteggio.

In [5]:
anno2num = dict()

with open('tesi.csv', encoding = 'utf-8') as infile:
  reader = csv.reader(infile)
  headers = next(reader)
  year_pos = headers.index('YEAR')
  for row in reader:
    year = row[year_pos]
    if year not in anno2num:
      anno2num[year] = 1
    else:
      anno2num[year] += 1

In [6]:
# ordiniamo per anno e stampiamo

for anno in sorted(anno2num):
    print(f"Nell'anno {anno} sono state fatte {anno2num[anno]} tesi")

Nell'anno 2020 sono state fatte 268 tesi
Nell'anno 2021 sono state fatte 290 tesi
Nell'anno 2022 sono state fatte 315 tesi
Nell'anno 2023 sono state fatte 277 tesi
Nell'anno 2024 sono state fatte 380 tesi
Nell'anno 2025 sono state fatte 415 tesi


## Scrivere

Il processo di scrittura è altrettanto semplice. Simuliamo la scrittura usando un oggetto `StringIO` che si comporta come un *file* ma in realtà è una stringa.

In [7]:
import io

header = ['YEAR', 'NUM']

with io.StringIO() as output:
  writer = csv.writer(output)
  writer.writerow(header)
  for anno in sorted(anno2num):
    writer.writerow([anno, str(anno2num[anno])])

  print(output.getvalue()) # stampiamo il contenuto del "file"

YEAR,NUM
2020,268
2021,290
2022,315
2023,277
2024,380
2025,415



Come esmepio, vediamo come "filtrare" un file CSV, scrivendo solo le righe che soddisfano una certa condizione. In questo caso, scriviamo solo le tesi in lingua spagnola.

In [8]:
with (
  open('tesi.csv', encoding = 'utf-8') as infile, 
  open('tesi-spa.csv', 'w', encoding = 'utf-8', newline ='') as outfile
  ):
  reader = csv.reader(infile)
  writer = csv.writer(outfile, delimiter=';', dialect='excel')
  headers = next(reader)
  language_pos = headers.index('LANGUAGE')
  writer.writerow(headers) # scriviamo le intestazioni
  for row in reader:
    if row[language_pos] == 'spa':
      writer.writerow(row)

# Leggere e scrivere JSON

Ora passiamo al formato JSON.

In [9]:
import json

## Leggere

Iniziamo col leggere i dati restituiti dalle API del chi e dove di Ateneo.

In [10]:
from urllib.request import urlopen

persona = 'santini' 
API_URL = 'https://apps.unimi.it/ws/chiedove/persona/fromNome/'

with urlopen(API_URL + persona) as response:
  chiedove = json.load(response) # il metodo load legge da un flusso

# le API trasmettono un dizionario, payLoad contiene una lista di risultati,
# vediamo le chiavi del primo risultato

list(chiedove['payLoad'][0].keys())

['altroTelefono',
 'matricola',
 'nome',
 'cognome',
 'email',
 'profiloPosizionePrimaria',
 'profiloPosizionePrimariaEng',
 'telefonoInterno',
 'urlChiEDove',
 'urlChiEDoveEng',
 'posizione',
 'posizionePrimaria',
 'posizionePrimariaEng',
 'fotoProfilo',
 'fotoProfiloDefault',
 'urlFotoProfilo']

## Scrivere

Scrivere, a partire da un oggetto, è altrettanto sempilce. Si può anche usare il metodo `dumps` che genera direttamente la stringa.

In [11]:
print(json.dumps(anno2num, indent=4)) # usiamolo sulla mappa tra anni e numero di tesi

{
    "2022": 315,
    "2024": 380,
    "2020": 268,
    "2023": 277,
    "2025": 415,
    "2021": 290
}


## Da oggetto a HTML

Talvolta il contenuto di un documento JSON è più complesso di un semplice dizionario. Ad esempio, la risposta del chi e dove contiene una lista di persone, ognuna con un certo numero di campi, alcuni dei quali a loro volta composti da altri dizionari. 

Per trasformare un dizioanrio in un documento HTML usiamo la **ricorsione**. L'idea è di trasformare ogni *list* in una tabella con una riga per ciascuna (tasformazione in HTML) dei suoi elementi, e ogni *dict* in una tabella con una riga per ciascuna coppia chiave-valore, in cui la chiave è l'intestazione della riga e (la trasformazione in HTML di) il valore è il contenuto della riga. 

In pseudo-codice:

<pre>
def to_html(obj):
  if obj è una lista:
    return una tabella con una riga contenente to_html(v) per ciascun elemento v di obj
  elif obj è un dizionario:  
    return una tabella con una riga contenente (k, to_html(v)) per ciascuna coppia k, v in obj
  else:
    return str(obj)
</pre>

Passando al codice vero e proprio avremo

In [12]:
def to_html(obj):
  res = []
  if isinstance(obj, list):
    res.append('<table class=json>')
    for v in obj:
      res.append(f'<tr><td>{to_html(v)}</td></tr>')
    res.append('</table>')
  elif isinstance(obj, dict):
    res.append('<table class=json>')
    for k, v in obj.items():
      res.append(f'<tr><th>{k}</th><td>{to_html(v)}</td></tr>')
    res.append('</table>')
  else:
    res.append(str(obj))
  return '\n'.join(res)    

In [13]:
# possiamo stampare la tabella ottenuta dalla mappa

print(to_html(anno2num))

<table class=json>
<tr><th>2022</th><td>315</td></tr>
<tr><th>2024</th><td>380</td></tr>
<tr><th>2020</th><td>268</td></tr>
<tr><th>2023</th><td>277</td></tr>
<tr><th>2025</th><td>415</td></tr>
<tr><th>2021</th><td>290</td></tr>
</table>


Oppure usare il metodo `HTML` e un po' di CSS per rendere più leggibile la tabella.

In [14]:
from IPython.display import HTML

STYLE = """<style>
.json { border-collapse: collapse !important; margin: .5em 0 !important; background: white !important; text-align: left !important; }
.json th { border: 1px solid #ccc !important; padding: 3px 8px !important; font-weight: bold !important; white-space: nowrap !important; background: white !important; text-align: left !important; }
.json td { border: 1px solid #ccc !important; padding: 3px 8px !important; background: white !important; text-align: left !important; }
</style>"""

HTML(STYLE + to_html(anno2num))

2022,315
2024,380
2020,268
2023,277
2025,415
2021,290


Ma possiamo anche scrivere il risultato su un file

In [15]:
with open('chiedove.html', 'w') as f:
  f.write(f'<html><head><title>Chiedove</title>{STYLE}</head><body>{to_html(chiedove)}</body></html>')